In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Banking Data Analysis Silver Layer").getOrCreate()

In [0]:
accounts = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Account_Data.csv", header = True, inferSchema = True)
branches = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Branch_Data.csv", header = True, inferSchema = True)
customers = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Customer_Data.csv", header = True, inferSchema = True)
loans = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Loan_Data.csv", header = True, inferSchema = True)
transactions = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Transacation_Data.csv", header = True, inferSchema = True)

In [0]:
display(accounts.limit(10))
display(branches.limit(10))
display(customers.limit(10))
display(loans.limit(10))
display(transactions.limit(10))

ACCOUNT_ID,CUSTOMER_ID,BRANCH_ID,OPENING_BALANCE,ACCOUNT_OPEN_DATE,ACCOUNT_TYPE,ACCOUNT_STATUS
A00001,null,b00019,520482,null,null,Pending
a00002,c00002,b00038,268824,2018-09-24,null,active
null,C00003,null,589218,2021-03-24,Savings,ACTIVE
a00004,null,b00021,202917,2017-11-15,Checking,CLOSED
null,c00005,b00049,426326,2015-11-28,SAVINGS,Closed
null,C00006,b00010,908163,2020-09-03,Fixed Deposit,Active
A00007,C00007,B00001,316236,2018-11-28,savings,ACTIVE
A00008,C00008,B00008,580292,2020-07-17,Fixed Deposit,Pending
a00009,C00009,null,796905,2016-06-10,fixed deposit,null
null,c00010,b00028,548251,2016-11-12,CHECKING,ACTIVE


BRANCH_ID,BRANCH_NAME,BRANCH_STATE
B00001,ND00000,North Dakota
B00002,NE00001,Nebraska
B00003,CT00002,Connecticut
B00004,MO00003,Missouri
B00005,NV00004,Nevada
B00006,ME00005,Maine
B00007,NY00006,New York
B00008,ME00007,Maine
B00009,null,Iowa
B00010,PA00009,null


CUSTOMER_ID,First_Name,Last_Name,City,Phone_Number,Occupation,DOB
null,timothy,nelson,davidville,1592081539,MAGICIAN,1915-09-03
null,Courtney,Farley,null,null,Scientist,1922-05-11
C00003,sheena,solis,new john,4311466522,BUSINESSMAN,1933-10-27
c00004,amy,Martinez,JONESTON,6112876134,MECHANIC,null
c00005,null,null,SOUTH CHRISTINE,8756770692,repairman,1999-03-01
C00006,Benjamin,Johnson,WEST EVANMOUTH,1402377116,Businessman,1942-04-21
C00007,CRAIG,ELLIOTT,sanchezshire,6202263306,Construction worker,1963-04-27
C00008,rebecca,KELLER,south christian,1063299332,Carpenter,1961-12-15
c00009,null,HOLT,CASEYTOWN,9923202932,priest,1951-12-23
C00010,Jeremy,LE,LAKE JUSTINMOUTH,5032107777,Paramedic,1961-12-31


LOAN_ID,CUSTOMER_ID,BRANCH_ID,LOAN_AMOUNT
L00001,C00341,B00033,6411642
L00002,C00344,B00009,7470012
L00003,C00551,B00028,3698741
L00004,C00974,B00004,null
L00005,C00097,B00022,null
L00006,null,null,8080934
L00007,null,B00046,null
L00008,C00837,B00029,null
L00009,C00592,B00045,9990503
L00010,C00068,B00031,6950491


TRANSCATION_ID,ACCOUNT_ID,TRANSCATION_DATE,TRANSCATION_MEDIA,TRANSCATION_TYPE,TRANSCATION_AMOUNT
T00001,null,2013-12-15,Credit_Card,Deposit,null
T00002,A00106,2018-08-21,Debit_Card,Transfer,null
T00003,A00233,2020-12-25,Check,Deposit,62526
T00004,A00848,2021-10-01,Credit_Card,Transfer,90070
T00005,A00404,2018-09-06,Cash,Deposit,80153
T00006,A00454,2016-06-21,Credit_Card,Purchase,83494
T00007,A00209,2019-02-01,Check,Deposit,38459
T00008,A00073,2022-12-16,Debit_Card,Withdrawal,18833
T00009,A00435,2023-06-21,Check,Deposit,68093
T00010,null,2013-10-31,Credit_Card,Deposit,40054


In [0]:
accounts.printSchema()
branches.printSchema()
customers.printSchema()
loans.printSchema()
transactions.printSchema()

root
 |-- ACCOUNT_ID: string (nullable = true)
 |-- CUSTOMER_ID: string (nullable = true)
 |-- BRANCH_ID: string (nullable = true)
 |-- OPENING_BALANCE: string (nullable = true)
 |-- ACCOUNT_OPEN_DATE: string (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ACCOUNT_STATUS: string (nullable = true)

root
 |-- BRANCH_ID: string (nullable = true)
 |-- BRANCH_NAME: string (nullable = true)
 |-- BRANCH_STATE: string (nullable = true)

root
 |-- CUSTOMER_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- DOB: string (nullable = true)

root
 |-- LOAN_ID: string (nullable = true)
 |-- CUSTOMER_ID: string (nullable = true)
 |-- BRANCH_ID: string (nullable = true)
 |-- LOAN_AMOUNT: string (nullable = true)

root
 |-- TRANSCATION_ID: string (nullable = true)
 |-- ACCOUNT_ID: string (nullable = 

## cleaning customers

In [0]:
median_phone = customers \
    .withColumn("Phone_Number",
        F.expr("try_cast(regexp_replace(Phone_Number, '[^0-9]', '') as BIGINT)")
    ) \
    .filter(F.col("Phone_Number").isNotNull()) \
    .approxQuantile("Phone_Number", [0.5], 0.01)[0]

In [0]:
clean_phone = F.expr(
    "try_cast(regexp_replace(Phone_Number, '[^0-9]', '') as BIGINT)"
)

silver_customers = (
    customers

    # 1. Drop rows where primary key is missing
    .filter(F.col("CUSTOMER_ID").isNotNull())

    # 2. Standardise CUSTOMER_ID
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))

    # 3. Names
    .withColumn("First_Name",
        F.when(F.col("First_Name").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("First_Name")))))
    .withColumn("Last_Name",
        F.when(F.col("Last_Name").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("Last_Name")))))

    # 4. City
    .withColumn("City",
        F.when(F.col("City").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("City")))))

    # 5. Clean + impute phone number 
    .withColumn("Phone_Number", clean_phone)
    .withColumn("Phone_Number",
        F.when(F.col("Phone_Number").isNull(), F.lit(median_phone))
         .otherwise(F.col("Phone_Number"))
    )

    # 6. Occupation
    .withColumn("Occupation",
        F.when(F.col("Occupation").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("Occupation")))))

    # 7. cast and fill nulls with default DOB 
    .withColumn("DOB",F.expr("try_to_date(DOB, 'yyyy-MM-dd')"))
    .withColumn("DOB",F.coalesce(F.col("DOB"), F.lit("1900-01-01").cast("date")))

    # 8. Dedup
    .dropDuplicates()
    .dropDuplicates(["CUSTOMER_ID"])

    # 9. Audit
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
display(silver_customers.limit(10))
silver_customers.printSchema()

CUSTOMER_ID,First_Name,Last_Name,City,Phone_Number,Occupation,DOB,_silver_loaded_at
C00010,Jeremy,Le,Lake Justinmouth,5.032107777E9,Paramedic,1961-12-31,2026-04-17T08:31:00.611Z
C00045,Benjamin,Jones,West Benjamin,2.781002995E9,Pirate,1927-08-22,2026-04-17T08:31:00.611Z
C00049,Jillian,Thornton,Frankfurt,7.951077572E9,Police Officer,1919-09-08,2026-04-17T08:31:00.611Z
C00056,Daniel,Joseph,Melvinport,3.014640268E9,Train Conductor,1992-05-28,2026-04-17T08:31:00.611Z
C00062,Kelly,Hudson,Unknown,3.066367311E9,Carpenter,1958-08-08,2026-04-17T08:31:00.611Z
C00094,Brittany,Padilla,Johnmouth,8.809642786E9,Pirate,1965-06-12,2026-04-17T08:31:00.611Z
C00104,Jasmine,Johnson,Kimberlyview,6.871658803E9,Farmer,1940-09-26,2026-04-17T08:31:00.611Z
C00105,Jon,Unknown,Brownside,9.885875169E9,Businessman,1956-10-02,2026-04-17T08:31:00.611Z
C00126,Molly,Thompson,Unknown,1.13078394E8,Pirate,1900-01-01,2026-04-17T08:31:00.611Z
C00152,Donna,Cooper,Lake Jeremiahport,3.253907221E9,Scientist,1900-01-01,2026-04-17T08:31:00.611Z


root
 |-- CUSTOMER_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone_Number: double (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- DOB: date (nullable = true)
 |-- _silver_loaded_at: timestamp (nullable = false)



## cleaning branch table

In [0]:
silver_branches = (
    branches

    # 1. Drop rows where primary key is missing
    .filter(F.col("BRANCH_ID").isNotNull())

    # 2. Standardise BRANCH_ID → uppercase + trim
    .withColumn("BRANCH_ID", F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Trim BRANCH_NAME; fill nulls
    .withColumn("BRANCH_NAME",
        F.when(F.col("BRANCH_NAME").isNull(), "Unknown Branch")
         .otherwise(F.trim(F.col("BRANCH_NAME"))))

    # 4. Proper-case BRANCH_STATE; fill nulls
    .withColumn("BRANCH_STATE",
        F.when(F.col("BRANCH_STATE").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("BRANCH_STATE")))))

    # 5. Drop fully duplicate rows, then dedup on primary key
    .dropDuplicates()
    .dropDuplicates(["BRANCH_ID"])

    # 6. Audit columns
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
print("Branches →", silver_branches.count())
display(silver_branches.limit(10))

Branches → 50


BRANCH_ID,BRANCH_NAME,BRANCH_STATE,_silver_loaded_at
B00009,Unknown Branch,Iowa,2026-04-17T08:30:44.373Z
B00008,ME00007,Maine,2026-04-17T08:30:44.373Z
B00038,MO00037,Missouri,2026-04-17T08:30:44.373Z
B00033,TX00032,Texas,2026-04-17T08:30:44.373Z
B00037,WI00036,Unknown,2026-04-17T08:30:44.373Z
B00039,CO00038,Colorado,2026-04-17T08:30:44.373Z
B00044,NC00043,North Carolina,2026-04-17T08:30:44.373Z
B00050,MO00049,Missouri,2026-04-17T08:30:44.373Z
B00002,NE00001,Nebraska,2026-04-17T08:30:44.373Z
B00004,MO00003,Missouri,2026-04-17T08:30:44.373Z


## cleaning accounts table

In [0]:
clean_balance = F.expr("""
    try_cast(
        regexp_replace(OPENING_BALANCE, '[^0-9.]', '') 
    as DOUBLE)
""")

median_balance = accounts \
    .withColumn("OPENING_BALANCE", clean_balance) \
    .filter(F.col("OPENING_BALANCE").isNotNull()) \
    .approxQuantile("OPENING_BALANCE", [0.5], 0.01)[0]

print(f"Median opening balance: {median_balance:,.0f}")

silver_accounts = (
    accounts

    # 1. Drop rows where primary key is missing
    .filter(F.col("ACCOUNT_ID").isNotNull())

    # 2. Standardise IDs → uppercase + trim
    .withColumn("ACCOUNT_ID",  F.upper(F.trim(F.col("ACCOUNT_ID"))))
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))
    .withColumn("BRANCH_ID",   F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Normalise ACCOUNT_TYPE: lowercase → title-case
    #    handles: "SAVINGS","savings","Savings" → "Savings"
    #             "FIXED DEPOSIT","fixed deposit" → "Fixed Deposit"
    .withColumn("ACCOUNT_TYPE",
        F.when(F.col("ACCOUNT_TYPE").isNull(), "Unknown")
         .otherwise(F.initcap(F.lower(F.trim(F.col("ACCOUNT_TYPE"))))))

    # 4. Normalise ACCOUNT_STATUS the same way
    #    handles: "ACTIVE","active","Active" → "Active"
    .withColumn("ACCOUNT_STATUS",
        F.when(F.col("ACCOUNT_STATUS").isNull(), "Unknown")
         .otherwise(F.initcap(F.lower(F.trim(F.col("ACCOUNT_STATUS"))))))

    # 5. Parse ACCOUNT_OPEN_DATE as DateType
    .withColumn("ACCOUNT_OPEN_DATE",F.when(F.col("ACCOUNT_OPEN_DATE").isin("null", "NULL", ""), None)
    .otherwise(F.expr("try_to_date(ACCOUNT_OPEN_DATE, 'yyyy-MM-dd')")))
    
    .withColumn("ACCOUNT_OPEN_DATE",
    F.coalesce(F.col("ACCOUNT_OPEN_DATE"), F.lit("1900-01-01").cast("date")))

    # 6. Impute OPENING_BALANCE with median; cast to FloatType
    .withColumn("OPENING_BALANCE", clean_balance)
    .withColumn("OPENING_BALANCE",F.when(F.col("OPENING_BALANCE").isNull(), F.lit(median_balance))
    .otherwise(F.col("OPENING_BALANCE")))

    # 7. Drop orphaned accounts (FK nulls — no parent customer/branch)
    .filter(F.col("CUSTOMER_ID").isNotNull())
    .filter(F.col("BRANCH_ID").isNotNull())

    # 8. Drop fully duplicate rows, then dedup on primary key
    .dropDuplicates()
    .dropDuplicates(["ACCOUNT_ID"])

    # 9. Audit columns
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

Median opening balance: 488,943


In [0]:
print("Accounts →", silver_accounts.count())
display(silver_accounts.limit(10))

Accounts → 983


ACCOUNT_ID,CUSTOMER_ID,BRANCH_ID,OPENING_BALANCE,ACCOUNT_OPEN_DATE,ACCOUNT_TYPE,ACCOUNT_STATUS,_silver_loaded_at
A00007,C00007,B00001,316236.0,2018-11-28,Savings,Active,2026-04-17T08:23:46.609Z
A00012,C00012,B00005,346915.0,2019-10-27,Fixed Deposit,Active,2026-04-17T08:23:46.609Z
A00044,C00044,B00034,760924.0,2017-10-10,Checking,Unknown,2026-04-17T08:23:46.609Z
A00111,C00111,B00006,495976.0,2014-05-13,Savings,Unknown,2026-04-17T08:23:46.609Z
A00112,C00112,B00011,416462.0,2015-11-19,Fixed Deposit,Pending,2026-04-17T08:23:46.609Z
A00114,C00114,B00014,852435.0,2019-06-26,Checking,Closed,2026-04-17T08:23:46.609Z
A00175,C00175,B00012,792562.0,2023-08-23,Savings,Closed,2026-04-17T08:23:46.609Z
A00176,C00176,B00003,488943.0,1900-01-01,Unknown,Closed,2026-04-17T08:23:46.609Z
A00189,C00189,B00007,488943.0,2023-03-18,Fixed Deposit,Pending,2026-04-17T08:23:46.609Z
A00202,C00202,B00043,166122.0,2019-05-15,Unknown,Closed,2026-04-17T08:23:46.609Z


## clean loans table

In [0]:
median_loan = loans \
    .withColumn("LOAN_AMOUNT", F.expr("try_cast(LOAN_AMOUNT as DOUBLE)")) \
    .filter(F.col("LOAN_AMOUNT").isNotNull()) \
    .approxQuantile("LOAN_AMOUNT", [0.5], 0.01)[0]

print(f"Median loan amount: {median_loan:,.0f}")

silver_loans = (
    loans

    # 1. Drop rows where primary key is missing
    .filter(F.col("LOAN_ID").isNotNull())

    # 2. Standardise IDs
    .withColumn("LOAN_ID",     F.upper(F.trim(F.col("LOAN_ID"))))
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))
    .withColumn("BRANCH_ID",   F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Clean + cast LOAN_AMOUNT safely
    .withColumn("LOAN_AMOUNT", F.expr("try_cast(LOAN_AMOUNT as DOUBLE)"))
    .withColumn("LOAN_AMOUNT", F.coalesce(F.col("LOAN_AMOUNT"), F.lit(median_loan)))

    # 4. Drop orphaned loans
    .filter(F.col("CUSTOMER_ID").isNotNull())
    .filter(F.col("BRANCH_ID").isNotNull())

    # 5. Remove duplicates
    .dropDuplicates()
    .dropDuplicates(["LOAN_ID"])

    # 6. Audit column (optional)
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

Median loan amount: 5,053,004


In [0]:
print("Loans →", silver_loans.count())
display(silver_loans.limit(10))

Loans → 488


LOAN_ID,CUSTOMER_ID,BRANCH_ID,LOAN_AMOUNT,_silver_loaded_at
L00002,C00344,B00009,7470012.0,2026-04-17T08:23:49.545Z
L00030,C00303,B00018,8146064.0,2026-04-17T08:23:49.545Z
L00034,C00733,B00047,5053004.0,2026-04-17T08:23:49.545Z
L00099,C00359,B00029,3699868.0,2026-04-17T08:23:49.545Z
L00125,C00093,B00001,375700.0,2026-04-17T08:23:49.545Z
L00148,C00336,B00002,9092574.0,2026-04-17T08:23:49.545Z
L00178,C00968,B00035,3369246.0,2026-04-17T08:23:49.545Z
L00181,C00264,B00041,5053004.0,2026-04-17T08:23:49.545Z
L00222,C00522,B00019,5053004.0,2026-04-17T08:23:49.545Z
L00267,C00673,B00028,8603024.0,2026-04-17T08:23:49.545Z


## clean transactions table

In [0]:
# ── Clean + compute median transaction amount ──────────────────────
median_txn = transactions \
    .withColumn("TRANSCATION_AMOUNT", F.expr("try_cast(TRANSCATION_AMOUNT as DOUBLE)")) \
    .filter(F.col("TRANSCATION_AMOUNT").isNotNull()) \
    .approxQuantile("TRANSCATION_AMOUNT", [0.5], 0.01)[0]

print(f"Median transaction amount: {median_txn:,.0f}")

# ── Window for latest record ───────────────────────────────
window_spec = Window.partitionBy("TRANSACTION_ID") \
                    .orderBy(F.col("TRANSACTION_DATE").desc())

silver_transactions = (
    transactions

    # 1. Drop null PK
    .filter(F.col("TRANSCATION_ID").isNotNull())

    # 2. Fix column names
    .withColumnRenamed("TRANSCATION_ID",     "TRANSACTION_ID")
    .withColumnRenamed("TRANSCATION_DATE",   "TRANSACTION_DATE")
    .withColumnRenamed("TRANSCATION_MEDIA",  "TRANSACTION_MEDIA")
    .withColumnRenamed("TRANSCATION_TYPE",   "TRANSACTION_TYPE")
    .withColumnRenamed("TRANSCATION_AMOUNT", "TRANSACTION_AMOUNT")

    # 3. Standardize IDs
    .withColumn("TRANSACTION_ID", F.upper(F.trim(F.col("TRANSACTION_ID"))))
    .withColumn("ACCOUNT_ID",     F.upper(F.trim(F.col("ACCOUNT_ID"))))

    #  4. Clean + parse DATE safely
    .withColumn("TRANSACTION_DATE",
        F.when(F.col("TRANSACTION_DATE").isin("null", "NULL", ""), None)
         .otherwise(F.expr("try_to_date(TRANSACTION_DATE, 'yyyy-MM-dd')"))
    )
    .withColumn("TRANSACTION_DATE",
        F.coalesce(F.col("TRANSACTION_DATE"), F.lit("1900-01-01").cast("date"))
    )

    #  5. Clean STRING columns
    # TRANSACTION_MEDIA
    .withColumn("TRANSACTION_MEDIA",
    F.when(
        (F.col("TRANSACTION_MEDIA").isNull()) | 
        (F.col("TRANSACTION_MEDIA").isin("null", "NULL", "")),
        "Unknown"
    ).otherwise(F.trim(F.col("TRANSACTION_MEDIA"))))

    # TRANSACTION_TYPE
    .withColumn("TRANSACTION_TYPE",
    F.when(
        (F.col("TRANSACTION_TYPE").isNull()) | 
        (F.col("TRANSACTION_TYPE").isin("null", "NULL", "")),
        "Unknown"
    ).otherwise(F.initcap(F.trim(F.col("TRANSACTION_TYPE")))))

    #  6. Clean + cast amount safely
    .withColumn("TRANSACTION_AMOUNT", F.expr("try_cast(TRANSACTION_AMOUNT as DOUBLE)"))
    .withColumn("TRANSACTION_AMOUNT",
        F.coalesce(F.col("TRANSACTION_AMOUNT"), F.lit(median_txn))
    )

    # 7. Drop orphan FK
    .filter(F.col("ACCOUNT_ID").isNotNull())

    #  8. Keep latest record per TRANSACTION_ID
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")

    # 9. Audit column (optional)
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

Median transaction amount: 49,713


In [0]:
print("Transactions →", silver_transactions.count())
display(silver_transactions.limit(10))

Transactions → 4838


TRANSACTION_ID,ACCOUNT_ID,TRANSACTION_DATE,TRANSACTION_MEDIA,TRANSACTION_TYPE,TRANSACTION_AMOUNT,_silver_loaded_at
T00001,A00527,2018-10-22,Credit_Card,Purchase,72450.0,2026-04-17T08:23:52.731Z
T00002,A00347,2021-03-29,Credit_Card,Deposit,10321.0,2026-04-17T08:23:52.731Z
T00003,A00233,2020-12-25,Check,Deposit,62526.0,2026-04-17T08:23:52.731Z
T00004,A00848,2021-10-01,Credit_Card,Transfer,90070.0,2026-04-17T08:23:52.731Z
T00005,A00404,2018-09-06,Cash,Deposit,80153.0,2026-04-17T08:23:52.731Z
T00006,A00454,2016-06-21,Credit_Card,Purchase,83494.0,2026-04-17T08:23:52.731Z
T00007,A00209,2019-02-01,Check,Deposit,38459.0,2026-04-17T08:23:52.731Z
T00008,A00073,2022-12-16,Debit_Card,Withdrawal,18833.0,2026-04-17T08:23:52.731Z
T00009,A00435,2023-06-21,Check,Deposit,68093.0,2026-04-17T08:23:52.731Z
T00011,A00356,2017-11-11,Debit_Card,Withdrawal,84020.0,2026-04-17T08:23:52.731Z


## Referential Integrity checks(check for Orphaned records)

In [0]:
checks = [
    ("account  → customer",  silver_accounts.join(silver_customers, "CUSTOMER_ID", "left_anti")),
    ("account  → branch",    silver_accounts.join(silver_branches,  "BRANCH_ID",   "left_anti")),
    ("loan     → customer",  silver_loans.join(silver_customers,    "CUSTOMER_ID", "left_anti")),
    ("loan     → branch",    silver_loans.join(silver_branches,     "BRANCH_ID",   "left_anti")),
    ("txn      → account",   silver_transactions.join(silver_accounts, "ACCOUNT_ID", "left_anti")),
]

print("\n=== Referential Integrity Report ===")
for label, orphans in checks:
    n = orphans.count()
    print(f"  {label:<25}  {'✓ OK' if n == 0 else f'⚠  {n} orphans'}")


=== Referential Integrity Report ===
  account  → customer        ⚠  1 orphans
  account  → branch          ✓ OK
  loan     → customer        ⚠  2 orphans
  loan     → branch          ✓ OK
  txn      → account         ⚠  80 orphans


## Referential Integrity Enforcement

In [0]:
# 1. Remove orphans from ACCOUNTS
silver_accounts_clean = (
    silver_accounts
    .join(silver_customers.select("CUSTOMER_ID"), "CUSTOMER_ID", "inner")
    .join(silver_branches.select("BRANCH_ID"),   "BRANCH_ID",   "inner")
)

print("Accounts after orphan removal →", silver_accounts_clean.count())
display(silver_accounts_clean.limit(10))

# 2. Remove orphans from LOANS
silver_loans_clean = (
    silver_loans
    .join(silver_customers.select("CUSTOMER_ID"), "CUSTOMER_ID", "inner")
    .join(silver_branches.select("BRANCH_ID"),   "BRANCH_ID",   "inner")
)

print("Loans after orphan removal →", silver_loans_clean.count())
display(silver_loans_clean.limit(10))

# 3. Remove orphans from TRANSACTIONS
silver_transactions_clean = (
    silver_transactions
    .join(silver_accounts_clean.select("ACCOUNT_ID"), "ACCOUNT_ID", "inner")
)

print("Transactions after orphan removal →", silver_transactions_clean.count())
display(silver_transactions_clean.limit(10))

Accounts after orphan removal → 982


BRANCH_ID,CUSTOMER_ID,ACCOUNT_ID,OPENING_BALANCE,ACCOUNT_OPEN_DATE,ACCOUNT_TYPE,ACCOUNT_STATUS,_silver_loaded_at
B00031,C00045,A00045,81828.0,2020-03-15,Checking,Pending,2026-04-17T08:30:09.822Z
B00014,C00049,A00049,488943.0,1900-01-01,Fixed Deposit,Unknown,2026-04-17T08:30:09.822Z
B00027,C00056,A00056,991902.0,2019-07-01,Checking,Pending,2026-04-17T08:30:09.822Z
B00016,C00062,A00062,415804.0,2021-09-30,Fixed Deposit,Closed,2026-04-17T08:30:09.822Z
B00005,C00094,A00094,103607.0,2018-09-14,Fixed Deposit,Unknown,2026-04-17T08:30:09.822Z
B00012,C00104,A00104,313146.0,2015-01-27,Fixed Deposit,Unknown,2026-04-17T08:30:09.822Z
B00005,C00105,A00105,148245.0,2022-05-25,Checking,Closed,2026-04-17T08:30:09.822Z
B00021,C00126,A00126,900462.0,2023-02-07,Savings,Pending,2026-04-17T08:30:09.822Z
B00047,C00152,A00152,488943.0,2023-02-13,Checking,Unknown,2026-04-17T08:30:09.822Z
B00007,C00179,A00179,698003.0,2023-05-23,Savings,Closed,2026-04-17T08:30:09.822Z


Loans after orphan removal → 486


BRANCH_ID,CUSTOMER_ID,LOAN_ID,LOAN_AMOUNT,_silver_loaded_at
B00009,C00344,L00002,7470012.0,2026-04-17T08:30:12.188Z
B00018,C00303,L00030,8146064.0,2026-04-17T08:30:12.188Z
B00047,C00733,L00034,5053004.0,2026-04-17T08:30:12.188Z
B00029,C00359,L00099,3699868.0,2026-04-17T08:30:12.188Z
B00001,C00093,L00125,375700.0,2026-04-17T08:30:12.188Z
B00002,C00336,L00148,9092574.0,2026-04-17T08:30:12.188Z
B00035,C00968,L00178,3369246.0,2026-04-17T08:30:12.188Z
B00041,C00264,L00181,5053004.0,2026-04-17T08:30:12.188Z
B00019,C00522,L00222,5053004.0,2026-04-17T08:30:12.188Z
B00028,C00673,L00267,8603024.0,2026-04-17T08:30:12.188Z


Transactions after orphan removal → 4749


ACCOUNT_ID,TRANSACTION_ID,TRANSACTION_DATE,TRANSACTION_MEDIA,TRANSACTION_TYPE,TRANSACTION_AMOUNT,_silver_loaded_at
A00007,T04161,2023-01-11,Unknown,Transfer,10846.0,2026-04-17T08:30:14.803Z
A00012,T04925,2017-04-27,Debit_Card,Unknown,90476.0,2026-04-17T08:30:14.803Z
A00044,T04595,2020-10-24,Cash,Transfer,25790.0,2026-04-17T08:30:14.803Z
A00111,T04216,2018-05-24,Credit_Card,Transfer,23755.0,2026-04-17T08:30:14.803Z
A00112,T04975,1900-01-01,Check,Transfer,85050.0,2026-04-17T08:30:14.803Z
A00114,T03573,2016-07-23,Credit_Card,Withdrawal,54745.0,2026-04-17T08:30:14.803Z
A00175,T03131,2023-03-05,Cash,Deposit,79641.0,2026-04-17T08:30:14.803Z
A00176,T04186,2020-04-30,Debit_Card,Purchase,16369.0,2026-04-17T08:30:14.803Z
A00189,T04921,2019-09-29,Debit_Card,Transfer,54497.0,2026-04-17T08:30:14.803Z
A00202,T04609,2023-08-22,Check,Deposit,39698.0,2026-04-17T08:30:14.803Z


In [0]:
# Check for Orphans AFTER Removal

checks_after = [
    ("account  → customer",  silver_accounts_clean.join(silver_customers, "CUSTOMER_ID", "left_anti")),
    ("account  → branch",    silver_accounts_clean.join(silver_branches,  "BRANCH_ID",   "left_anti")),
    ("loan     → customer",  silver_loans_clean.join(silver_customers,    "CUSTOMER_ID", "left_anti")),
    ("loan     → branch",    silver_loans_clean.join(silver_branches,     "BRANCH_ID",   "left_anti")),
    ("txn      → account",   silver_transactions_clean.join(silver_accounts_clean, "ACCOUNT_ID", "left_anti")),
]

print("\n=== Referential Integrity AFTER Removal ===")
for label, orphans in checks_after:
    n = orphans.count()
    print(f"  {label:<25}  {'✓ OK (0 orphans)' if n == 0 else f'⚠ {n} orphans still exist'}")


=== Referential Integrity AFTER Removal ===
  account  → customer        ✓ OK (0 orphans)
  account  → branch          ✓ OK (0 orphans)
  loan     → customer        ✓ OK (0 orphans)
  loan     → branch          ✓ OK (0 orphans)
  txn      → account         ✓ OK (0 orphans)
